In [20]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/kgvsfcekman/ekman6_vsfc_2.rdf
/kaggle/input/kgvsfcekman/ekman6_vsfc_4.rdf
/kaggle/input/kgvsfcekman/ekman6_2_vsfc_3.rdf
/kaggle/input/kgvsfcekman/ekman6_vsfc_weighted.rdf
/kaggle/input/kgvsfcekman/ekman6_4_4.rdf
/kaggle/input/kgvsfcekman/ekman6_vsfc_3.rdf
/kaggle/input/kgvsfcekman/ekman6_4_6.rdf
/kaggle/input/kgvsfcekman/ekman6_3_vsfc_5.rdf
/kaggle/input/kgvsfcekman/ekman6_3_vsfc_4.rdf
/kaggle/input/kgvsfcekman/ekman6_3_vsfc_1.rdf
/kaggle/input/kgvsfcekman/ekman6_4_3.rdf
/kaggle/input/kgvsfcekman/ekman6_vsfc.rdf
/kaggle/input/uit-vsfc/UIT-VSFC/README.txt
/kaggle/input/uit-vsfc/UIT-VSFC/test/topics.txt
/kaggle/input/uit-vsfc/UIT-VSFC/test/sentiments.txt
/kaggle/input/uit-vsfc/UIT-VSFC/test/sents.txt
/kaggle/input/uit-vsfc/UIT-VSFC/train/topics.txt
/kaggle/input/uit-vsfc/UIT-VSFC/train/sentiments.txt
/kaggle/input/uit-vsfc/UIT-VSFC/train/sents.txt
/kaggle/input/uit-vsfc/UIT-VSFC/dev/topics.txt
/kaggle/input/uit-vsfc/UIT-VSFC/dev/sentiments.txt
/kaggle/input/uit-vsfc/UIT-VSF

In [21]:
import random
import numpy as np
import torch
import os

def seed_everything(seed=2024):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Thiết lập seed cố định
seed_everything(123)
print("✅ Đã thiết lập Random Seed: (Kết quả sẽ cố định mỗi lần chạy)")

✅ Đã thiết lập Random Seed: (Kết quả sẽ cố định mỗi lần chạy)


In [22]:
# =========================================================
# CELL 2: ONTOLOGY ENGINE V12.3 (VSFC BOOSTED EDITION)
# Logic: Deep Inference (giống VSMEC) + Polarity Boosting.
# Target: Ép tín hiệu Polarity mạnh lên để hỗ trợ VSFC.
# =========================================================
!pip install -q pyvi rdflib unidecode

import os, re
import numpy as np
import rdflib
from rdflib import Graph, Namespace
from rdflib.namespace import RDF, RDFS, OWL
from pyvi import ViTokenizer
from unidecode import unidecode

class OntologyEngine:
    def __init__(self, rdfpath, defaultconf=0.85, manual_boost=1.2, negation_attenuation=0.5, alpha_similarity=0.2, polarity_boost=2.0, verbose=True):
        self.EKMAN = Namespace("http://example.org/ekman-ontology#")
        self.defaultconf = float(defaultconf)
        self.manual_boost = float(manual_boost)
        self.negation_attenuation = float(negation_attenuation) 
        self.alpha_similarity = float(alpha_similarity)
        self.polarity_boost = float(polarity_boost) # [NEW] Hệ số kích đại cực tính
        self.verbose = bool(verbose)

        # 1. CONFIGS
        self.ALLEMOTIONS = ["Anger", "Disgust", "Fear", "Happiness", "Neutral", "Sadness", "Surprise"]
        self.ALLAPPRAISALS = [
            "GoalObstructionAppraisal", "PleasantnessAppraisal", "UnpleasantnessAppraisal", 
            "DangerousnessAppraisal", "LossAppraisal", "SuddennessAppraisal", 
            "LowPredictabilityAppraisal", "UnpredictabilityAppraisal", "UnfairnessAppraisal"
        ]
        self.INTENSITY_CLASSES = ["HighIntensity", "MediumIntensity", "LowIntensity"]
        self.POLARITY_CLASSES = ["PositivePolarity", "NegativePolarity", "NeutralPolarity"]

        self.EMOTIONMAPPING = {
            "Enjoyment": "Happiness", "Happiness": "Happiness", "Anger": "Anger", "Disgust": "Disgust", 
            "Fear": "Fear", "Sadness": "Sadness", "Surprise": "Surprise", 
            "NeutralEmotion": "Neutral", "Neutral": "Neutral", "Other": "Neutral", "NegationCue": "NegationCue"
        }
        
        # [TUNING CHO VSFC]
        self.IMPLIED_POLARITY = {
            # Emotion -> Polarity
            "Anger": "NegativePolarity", "Disgust": "NegativePolarity",
            "Fear": "NegativePolarity", "Sadness": "NegativePolarity",
            "Happiness": "PositivePolarity", "Enjoyment": "PositivePolarity",
            "Surprise": "NeutralPolarity", # [FIX] VSFC: Surprise (bất ngờ) chưa chắc là Pos -> Để Neutral an toàn hơn
            # Appraisal -> Polarity (Vũ khí bí mật của V12.2)
            "UnpleasantnessAppraisal": "NegativePolarity",
            "GoalObstructionAppraisal": "NegativePolarity", 
            "LossAppraisal": "NegativePolarity",
            "UnfairnessAppraisal": "NegativePolarity",
            "DangerousnessAppraisal": "NegativePolarity",
            "PleasantnessAppraisal": "PositivePolarity"
        }
        
        self.AMBIGUOUS_TERMS = {
            "hay", "chả", "cơ", "ngờ", "ý", "quá", "lại", 
            "tôi", "tao", "tớ", "mình", "bạn", "nó", "hắn", "anh", "chị", "em",
            "là", "thì", "mà", "bị", "được", "cái", "con", "người"
        }
        
        self.VALID_DOMAINS = {"GeneralDomain", "SocialDomain", "EducationDomain"}
        
        # LOAD KG
        if self.verbose: print(f"📥 Loading KG for VSFC: {rdfpath}")
        self.g = Graph()
        if os.path.exists(rdfpath):
            self.g.parse(rdfpath)
            if self.verbose: print(f"   ✅ Loaded {len(self.g)} triples.")
        else:
            print(f"   ⚠️ ERROR: File not found {rdfpath}")

        self.sim_matrix = self._build_similarity_matrix()
        self.lexiconmap = self._build_strict_lexicon()
        
        self.mwe_max_len = 1
        for k in self.lexiconmap.keys(): self.mwe_max_len = max(self.mwe_max_len, len(k.split()))
        if self.verbose: print(f"   📚 Lexicon size: {len(self.lexiconmap)} entries (v12.3 VSFC-Boosted)")

    def _norm(self, s): return re.sub(r"\s+", " ", str(s).lower().strip().replace("_", " ")).strip() if s else ""
    def _local(self, uri): return str(uri).split("#")[-1] if "#" in str(uri) else str(uri).split("/")[-1]

    def _build_similarity_matrix(self):
        n = len(self.ALLEMOTIONS)
        mat = np.zeros((n, n), dtype=np.float32); np.fill_diagonal(mat, 1.0)
        try:
            q = """PREFIX ekman: <http://example.org/ekman-ontology#> SELECT ?score ?e1 ?e2 WHERE { ?edge a ekman:EmotionSimilarityEdge ; ekman:similarityScore ?score ; ekman:linksEmotion ?e1 ; ekman:linksEmotion ?e2 . }"""
            for row in self.g.query(q):
                e1, e2 = self.EMOTIONMAPPING.get(self._local(row.e1)), self.EMOTIONMAPPING.get(self._local(row.e2))
                if e1 in self.ALLEMOTIONS and e2 in self.ALLEMOTIONS:
                    i, j = self.ALLEMOTIONS.index(e1), self.ALLEMOTIONS.index(e2)
                    mat[i, j] = mat[j, i] = max(mat[i, j], float(row.score))
        except: pass
        s = mat.sum(axis=1, keepdims=True); s[s==0] = 1.0
        return mat / s

    def _build_strict_lexicon(self):
        EK, lookup = self.EKMAN, {}
        mix_cache, sim_cache = {}, {}
        
        for r in self.g.query("PREFIX ekman: <http://example.org/ekman-ontology#> SELECT ?m ?e ?w WHERE { ?m a ekman:EmotionMixture ; ekman:hasComponent ?c . ?c ekman:componentEmotion ?e ; ekman:componentWeight ?w . }"):
            m, e, w = r.m, self.EMOTIONMAPPING.get(self._local(r.e)), float(r.w)
            if e in self.ALLEMOTIONS: mix_cache.setdefault(m, {})[e] = mix_cache.setdefault(m, {}).get(e, 0.0) + w
        
        for s, _, o in self.g.triples((None, EK.hasEmotionCategory, None)):
            e = self.EMOTIONMAPPING.get(self._local(o))
            if e in self.ALLEMOTIONS: sim_cache[s] = [e]
            
        for s in self.g.subjects(RDF.type, None):
            if s in mix_cache or s in sim_cache: continue
            ts = list(self.g.objects(s, RDF.type))
            if EK.NegationCue in ts: sim_cache[s] = ["NegationCue"]
            else:
                for t in ts:
                    m = self.EMOTIONMAPPING.get(self._local(t))
                    if m in self.ALLEMOTIONS: sim_cache[s] = [m]; break
        
        q = """PREFIX ekman: <http://example.org/ekman-ontology#> SELECT ?word ?stim ?ev ?base ?type ?dom WHERE { ?evoc a ekman:Evocation ; ekman:fromLexiconEntry ?lex ; ekman:toStimulus ?stim . ?lex ekman:hasContent ?word . OPTIONAL { ?evoc ekman:confidenceScore ?ev } OPTIONAL { ?evoc ekman:evidenceType ?type } OPTIONAL { ?lex ekman:baseIntensityScore ?base } OPTIONAL { ?lex ekman:belongsToDomain ?dom } }"""
        for r in self.g.query(q):
            if r.dom and self._local(r.dom) not in self.VALID_DOMAINS: continue 
            w = self._norm(r.word)
            if len(w) < 2 or w in self.AMBIGUOUS_TERMS: continue
            
            emos = {}
            if r.stim in mix_cache:
                raw = mix_cache[r.stim]; tot = sum(raw.values())
                if tot > 0: emos = {k: v/tot for k, v in raw.items()}
            elif r.stim in sim_cache:
                lst = sim_cache[r.stim]
                emos = {k: 1.0 for k in lst}
            if not emos: continue

            apps, ints, pols = self._infer_attributes(r.stim, emos.keys())
            
            entry_score = float(r.ev or self.defaultconf) * float(r.base or 1.0)
            
            entry = {"emotions": emos, "score": entry_score, 
                     "appraisals": list(apps), "intensities": list(ints), "polarities": list(pols), 
                     "isnegation": "NegationCue" in emos}
            lookup.setdefault(w, []).append(entry)
            wu = self._norm(unidecode(w))
            if wu != w: lookup.setdefault(wu, []).append(entry)
        return lookup

    def _infer_attributes(self, s, emos):
        EK, a, i, p = self.EKMAN, set(), set(), set()
        
        def g(r):
            for o in self.g.objects(r, EK.evokesAppraisal): a.add(self._local(o))
            for o in self.g.objects(r, EK.hasIntensity): i.add(self._local(o))
            for o in self.g.objects(r, EK.hasPolarity): p.add(self._local(o))
        g(s)
        for t in self.g.objects(s, RDF.type): g(t)
        
        # Deep Inference (Logic V12.2)
        if not p:
            for e in emos:
                if e in self.IMPLIED_POLARITY: p.add(self.IMPLIED_POLARITY[e])
        if not p:
            for app in a:
                if app in self.IMPLIED_POLARITY: p.add(self.IMPLIED_POLARITY[app])

        return a, i, p

    def getvector(self, text, debug=False):
        tokens = ViTokenizer.tokenize(str(text).lower()).split()
        flat = [self._norm(t) for t in tokens if self._norm(t)]
        
        ve, va, vi = np.zeros(7, dtype=np.float32), np.zeros(9, dtype=np.float32), np.zeros(3, dtype=np.float32)
        vp, vn = np.zeros(3, dtype=np.float32), np.zeros(2, dtype=np.float32)
        
        idx, n = 0, len(flat)
        hit_trace = []

        while idx < n:
            m, ml = None, 0
            for L in range(min(self.mwe_max_len, n-idx), 0, -1):
                p = " ".join(flat[idx:idx+L])
                if p in self.lexiconmap: m, ml = self.lexiconmap[p], L; break
            
            if not m:
                idx += 1; continue
            
            if debug: hit_trace.append({"phrase": " ".join(flat[idx:idx+ml]), "data": m[0]})

            for ent in m:
                if ent["isnegation"]:
                    vn[0] = 1.0; continue
                
                sc = ent["score"]
                if vn[0] > 0: sc *= self.negation_attenuation 
                
                for e, w in ent["emotions"].items():
                    if e in self.ALLEMOTIONS: ve[self.ALLEMOTIONS.index(e)] += sc * w
                for x in ent["appraisals"]:
                    if x in self.ALLAPPRAISALS: va[self.ALLAPPRAISALS.index(x)] += sc
                for x in ent["intensities"]:
                    if x in self.INTENSITY_CLASSES: vi[self.INTENSITY_CLASSES.index(x)] = 1.0
                
                # [QUAN TRỌNG V12.3] BOOST POLARITY
                # Nhân thêm hệ số polarity_boost để tín hiệu Sentiment áp đảo Emotion
                for x in ent["polarities"]:
                    if x in self.POLARITY_CLASSES: 
                        vp[self.POLARITY_CLASSES.index(x)] += sc * self.polarity_boost 
            idx += ml

        if self.alpha_similarity > 0: ve = np.clip(ve + self.alpha_similarity * (ve @ self.sim_matrix), 0, None)
        fin = np.concatenate([np.tanh(ve), np.tanh(va), vi, vp, vn]).astype(np.float32)
        return (fin, flat, hit_trace) if debug else fin

# KHỞI TẠO (Dùng file 6_4_8)
RDFPATH = "/kaggle/input/kg-ekman6/ekman6_4_8.rdf"
if os.path.exists(RDFPATH):
    print(f"🚀 Initializing OntologyEngine v12.3 (VSFC Boosted) with {RDFPATH}...")
    ontengine = OntologyEngine(RDFPATH, polarity_boost=2.0, verbose=True) # Boost = 2.0
    
    # Test thử xem điểm Negative có cao vọt lên không
    v, _, _ = ontengine.getvector("tệ hại", debug=True)
    print(f"\n🧪 BOOST CHECK: Anger={v[0]:.2f}, Negative={v[20]:.2f}")
    if v[20] > v[0]: print("✅ Polarity signal is DOMINANT now!")
else:
    print(f"❌ ERROR: File not found {RDFPATH}")

🚀 Initializing OntologyEngine v12.2 (Deep Inference) with /kaggle/input/kg-ekman6/ekman6_4_8.rdf...
📥 Loading KG: /kaggle/input/kg-ekman6/ekman6_4_8.rdf
   ✅ Loaded 90420 triples.
   📚 Lexicon size: 2540 entries (v12.2 Deep-Inference)

🧪 CHECK POLARITY 'tệ hại':
   -> Negative Score in Vector: 0.0 (Success if > 0)


In [23]:
v, _, matches = ontengine.getvector("thất vọng", debug=True)
print("Vector Neg:", v[20]) # Index 20 là NegativePolarity
if matches: print("Matched Data:", matches[0]['data']['polarities'])

Vector Neg: 0.5475
Matched Data: ['NegativePolarity']


In [24]:
def test_vector_24d(text, engine):
    print(f"--- ANALYZING: '{text}' ---")
    
    # ---------------------------------------------------------
    # SỬA LỖI: Dùng getvector với debug=True thay vì get_matches
    # Output trả về: (Vector kết quả, Danh sách Token, Trace các từ match được)
    # ---------------------------------------------------------
    vector, tokens, matches = engine.getvector(text, debug=True)
    
    print(f"\n1. TOKENS: {tokens}")
    
    print(f"\n2. MATCHED PHRASES (Lexicon Hits):")
    if not matches:
        print("   (No phrases found in Ontology)")
    else:
        for m in matches:
            # m cấu trúc: {'phrase': 'từ tìm thấy', 'data': {chi tiết điểm số...}}
            print(f"   ✅ Phrase: '{m['phrase']}'")
            print(f"      -> Is Negation? {m['data']['isnegation']}")
            print(f"      -> Score: {m['data']['score']}")
            print(f"      -> Emotions: {m['data']['emotions']}")

    print(f"\n3. FINAL VECTOR SHAPE: {vector.shape}")
    print(f"   (Vector này sẽ được nối vào PhoBERT)")
    
    print(f"\n4. EMOTION VALUES (7 Dimensions):")
    # Mapping lại tên để dễ nhìn
    labels = ["Anger", "Disgust", "Fear", "Happiness", "Neutral", "Sadness", "Surprise"]
    for i, label in enumerate(labels):
        print(f"   - {label}: {vector[i]:.4f}")

# --- CHẠY THỬ LẠI ---
test_text = "Món ăn này không ngon lắm nhưng phục vụ rất tuyệt vời"
if 'ontengine' in globals():
    test_vector_24d(test_text, ontengine)
else:
    print("⚠️ Hãy chạy Cell khởi tạo OntologyEngine trước!")

--- ANALYZING: 'Món ăn này không ngon lắm nhưng phục vụ rất tuyệt vời' ---

1. TOKENS: ['món', 'ăn', 'này', 'không', 'ngon', 'lắm', 'nhưng', 'phục vụ', 'rất', 'tuyệt vời']

2. MATCHED PHRASES (Lexicon Hits):
   ✅ Phrase: 'không'
      -> Is Negation? True
      -> Score: 0.95
      -> Emotions: {'NegationCue': 1.0}
   ✅ Phrase: 'rất'
      -> Is Negation? False
      -> Score: 0.372
      -> Emotions: {'Happiness': 1.0}
   ✅ Phrase: 'tuyệt vời'
      -> Is Negation? False
      -> Score: 0.5775
      -> Emotions: {'Happiness': 1.0}

3. FINAL VECTOR SHAPE: (24,)
   (Vector này sẽ được nối vào PhoBERT)

4. EMOTION VALUES (7 Dimensions):
   - Anger: 0.0000
   - Disgust: 0.0000
   - Fear: 0.0000
   - Happiness: 0.5151
   - Neutral: 0.0000
   - Sadness: 0.0000
   - Surprise: 0.0000


In [25]:
# ---------------------------------------------------------
# CELL 3: DATASET & PREPROCESSING (UIT-VSFC)
# ---------------------------------------------------------
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from sklearn.utils.class_weight import compute_class_weight
import os

# 1. Khởi tạo Tokenizer
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")
BASE_PATH = "/kaggle/input/uit-vsfc/UIT-VSFC"

def load_vsfc_folder(folder_name):
    """Hàm đọc dữ liệu chuẩn VSFC"""
    sent_path = f"{BASE_PATH}/{folder_name}/sents.txt"
    label_path = f"{BASE_PATH}/{folder_name}/sentiments.txt"
    try:
        with open(sent_path, 'r', encoding='utf-8') as f:
            sentences = [line.strip() for line in f.readlines()]
        with open(label_path, 'r', encoding='utf-8') as f:
            # 0: Negative, 1: Neutral, 2: Positive
            labels = [int(line.strip()) for line in f.readlines()]
        
        min_len = min(len(sentences), len(labels))
        return pd.DataFrame({'Sentence': sentences[:min_len], 'Sentiment': labels[:min_len]})
    except FileNotFoundError:
        print(f"❌ Không tìm thấy thư mục {folder_name} tại {BASE_PATH}")
        # Trả về DataFrame rỗng để không crash code
        return pd.DataFrame({'Sentence': [], 'Sentiment': []})

# 2. Load Data
df_train = load_vsfc_folder("train")
df_valid = load_vsfc_folder("dev")
df_test = load_vsfc_folder("test")

print(f"✅ Loaded: Train={len(df_train)}, Dev={len(df_valid)}, Test={len(df_test)}")

# Kiểm tra xem data có load được không
if len(df_train) == 0:
    print("⚠️ CẢNH BÁO: Không load được dữ liệu. Hãy kiểm tra lại đường dẫn BASE_PATH!")
else:
    class VSFC_OntologyDataset(Dataset):
        def __init__(self, df, tokenizer, engine, max_len=128):
            self.df = df
            self.tokenizer = tokenizer
            self.engine = engine
            self.max_len = max_len
            
        def __len__(self): return len(self.df)
        
        def __getitem__(self, idx):
            txt = str(self.df.iloc[idx]['Sentence'])
            # Đảm bảo nhãn là int
            lbl = int(self.df.iloc[idx]['Sentiment']) 
            
            enc = self.tokenizer.encode_plus(
                txt, max_length=self.max_len, padding='max_length', 
                truncation=True, return_tensors='pt'
            )
            
            # Gọi Engine để lấy vector KG
            vec = self.engine.getvector(txt)
            
            return {
                'input_ids': enc['input_ids'].flatten(),
                'attention_mask': enc['attention_mask'].flatten(),
                'ontology_features': torch.tensor(vec, dtype=torch.float),
                'labels': torch.tensor(lbl, dtype=torch.long)
            }

    # DataLoaders
    # Lưu ý: ontengine phải được khởi tạo từ Cell 2 trước đó
    if 'ontengine' in globals():
        train_loader = DataLoader(VSFC_OntologyDataset(df_train, tokenizer, ontengine), batch_size=32, shuffle=True)
        valid_loader = DataLoader(VSFC_OntologyDataset(df_valid, tokenizer, ontengine), batch_size=32)
        test_loader = DataLoader(VSFC_OntologyDataset(df_test, tokenizer, ontengine), batch_size=32)

        # Class Weights
        train_labels = df_train['Sentiment'].tolist()
        cls_w = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
        
        # Đưa về GPU nếu có
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        weights_tensor = torch.tensor(cls_w, dtype=torch.float).to(device)
        print(f"⚖️ Class Weights (Neg, Neu, Pos): {weights_tensor}")
    else:
        print("❌ LỖI: Biến 'ontengine' chưa tồn tại. Vui lòng chạy lại Cell 2!")

✅ Loaded: Train=11426, Dev=1583, Test=3166
⚖️ Class Weights (Neg, Neu, Pos): tensor([0.7152, 8.3159, 0.6749], device='cuda:0')


In [26]:
# =========================================================
# CELL: INSPECT VECTOR DISTRIBUTION (DATAFRAME VIEW)
# Mục tiêu: Liệt kê vector của nhiều câu dưới dạng bảng để dễ so sánh.
# =========================================================
import pandas as pd
import numpy as np

# 1. Định nghĩa tên cột cho 24 chiều
COLS_24D = (
    ["Anger", "Disgust", "Fear", "Happiness", "Neutral", "Sadness", "Surprise"] + # 0-6
    ["Pleas", "Unpleas", "Loss", "Danger", "Unpred", "Sudden", "GoalObs", "Unfair", "Other"] + # 7-15
    ["Int_Hi", "Int_Me", "Int_Lo"] + # 16-18
    ["Pos", "Neg", "Neu"] + # 19-21
    ["NegHit", "NegCnt"] # 22-23
)

def inspect_vectors(texts, engine):
    data = []
    for txt in texts:
        # Lấy vector
        vec = engine.getvector(txt)
        # Làm tròn để dễ nhìn
        vec_rounded = np.round(vec, 2)
        
        # Tạo row
        row = {"Sentence": txt}
        for i, val in enumerate(vec_rounded):
            # Chỉ lưu các giá trị khác 0 để bảng gọn (hoặc lưu hết tuỳ ý)
            # Ở đây lưu hết để thấy cấu trúc
            row[COLS_24D[i]] = val
        data.append(row)
    
    # Tạo DataFrame
    df_results = pd.DataFrame(data)
    
    # Style: Tô màu các ô khác 0 để nổi bật
    def highlight_nonzero(val):
        color = '#d1e7dd' if isinstance(val, (int, float)) and abs(val) > 0.01 else '' # Màu xanh nhạt
        if isinstance(val, (int, float)) and val > 0.8: color = '#198754; color: white' # Xanh đậm nếu score cao
        if isinstance(val, (int, float)) and val < -0.5: color = '#dc3545; color: white' # Đỏ nếu âm (hiếm gặp vì tanh)
        return f'background-color: {color}'

    return df_results.style.map(highlight_nonzero, subset=COLS_24D)

# --- A. TEST TRÊN CÁC TRƯỜNG HỢP CỤ THỂ ---
samples = [
    "Món này ngon tuyệt vời",                  # Positive + Patch
    "Thái độ phục vụ quá tệ hại",             # Negative + Patch + Intensifier
    "Không ngon lắm",                         # Negation + Intensifier
    "Món ăn bình thường, không có gì đặc sắc", # Mixed / Neutral
    "Tôi rất thất vọng về cách làm việc",     # Sadness + Patch
    "Sợ quá đi mất",                          # Fear + Intensifier
    "Tôi chẳng thấy buồn cười gì cả"          # Negation strong
]

print(">>> A. MANUAL SAMPLES INSPECTION:")
display(inspect_vectors(samples, ontengine))

# --- B. TEST TRÊN DATASET THỰC TẾ (Nếu đã load df_train) ---
if 'df_train' in globals() and len(df_train) > 0:
    print("\n>>> B. RANDOM SAMPLES FROM TRAINING DATA:")
    random_samples = df_train['Sentence'].sample(5).tolist()
    display(inspect_vectors(random_samples, ontengine))
else:
    print("\n(Chưa load df_train nên bỏ qua phần B)")

>>> A. MANUAL SAMPLES INSPECTION:


,Sentence,Anger,Disgust,Fear,Happiness,Neutral,Sadness,Surprise,Pleas,Unpleas,Loss,Danger,Unpred,Sudden,GoalObs,Unfair,Other,Int_Hi,Int_Me,Int_Lo,Pos,Neg,Neu,NegHit,NegCnt
0,Món này ngon tuyệt vời,0.000000,0.000000,0.000000,0.600000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.580000,0.000000,0.000000,0.000000,0.000000
1,Thái độ phục vụ quá tệ hại,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,Không ngon lắm,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
3,"Món ăn bình thường, không có gì đặc sắc",0.000000,0.000000,0.000000,0.000000,0.640000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
4,Tôi rất thất vọng về cách làm việc,0.000000,0.000000,0.000000,0.420000,0.000000,0.580000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.370000,0.550000,0.000000,0.000000,0.000000
5,Sợ quá đi mất,0.000000,0.000000,0.750000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.810000,0.000000,0.000000,0.000000
6,Tôi chẳng thấy buồn cười gì cả,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000



>>> B. RANDOM SAMPLES FROM TRAINING DATA:


,Sentence,Anger,Disgust,Fear,Happiness,Neutral,Sadness,Surprise,Pleas,Unpleas,Loss,Danger,Unpred,Sudden,GoalObs,Unfair,Other,Int_Hi,Int_Me,Int_Lo,Pos,Neg,Neu,NegHit,NegCnt
0,có khả năng truyền đạt tốt .,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,"khi có nhóm lên seminar thì phải có ít nhất ba câu hỏi thì mới "" đậu "" , không cần biết câu hỏi là gì , không đủ ba câu thì phải làm lại .",0.600000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.580000,0.000000,1.000000,0.000000
2,"thầy giảng buồn ngủ , không nhiệt tình , bài tập thực hành không có .",0.000000,0.550000,0.000000,0.330000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.290000,0.510000,0.000000,1.000000,0.000000
3,sữa lỗi cho sinh viên .,0.330000,0.000000,0.000000,0.000000,0.000000,0.390000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.630000,0.000000,0.000000,0.000000
4,phòng thực hành chưa có chất lượng .,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [27]:
# ---------------------------------------------------------
# CELL 4: MODEL ARCHITECTURE (FIXED IMPORTS)
# ---------------------------------------------------------
import torch
import torch.nn as nn  # <--- ĐÃ THÊM DÒNG NÀY ĐỂ SỬA LỖI
from transformers import AutoModel

# Tự động lấy chiều dài vector đầu vào
try: 
    # Test thử vector dummy để lấy kích thước thật
    dummy_vec = ontengine.getvector("test")
    # Nếu đang ở chế độ debug nó trả về tuple, lấy phần tử đầu
    if isinstance(dummy_vec, tuple): dummy_vec = dummy_vec[0]
    ONT_INPUT_DIM = len(dummy_vec)
    print(f"📏 Detected Ontology Dimension: {ONT_INPUT_DIM}")
except: 
    ONT_INPUT_DIM = 24 # Fallback mặc định
    print(f"⚠️ Could not detect dim, using default: {ONT_INPUT_DIM}")

class PhoBERT_Fusion_V2(nn.Module):
    def __init__(self, n_classes, fusion_type='none', ontology_dim=None, ont_input_dim=ONT_INPUT_DIM):
        super(PhoBERT_Fusion_V2, self).__init__()
        self.fusion_type = fusion_type
        
        # Backbone
        self.phobert = AutoModel.from_pretrained("vinai/phobert-base-v2")
        self.dropout = nn.Dropout(p=0.3)
        
        # Ontology Adapter
        if ontology_dim: # Dense Mode (ví dụ 64d)
            self.ont_adapter = nn.Sequential(
                nn.Linear(ont_input_dim, ontology_dim),
                nn.BatchNorm1d(ontology_dim),
                nn.ReLU(),
                nn.Dropout(0.1)
            )
            self.curr_dim = ontology_dim
        else: # Raw Mode (xAI - 24d)
            self.ont_adapter = nn.Identity()
            self.curr_dim = ont_input_dim
            
        # Gating Mechanism (Nếu dùng Gate)
        if self.fusion_type == 'gate':
            # Học cách lọc thông tin từ Text
            self.gate = nn.Linear(768, self.curr_dim)
            
        # Classifier Head
        # Nếu concat/gate thì input size tăng thêm, nếu none thì giữ nguyên 768
        inp_dim = 768 + self.curr_dim if self.fusion_type != 'none' else 768
        self.fc = nn.Linear(inp_dim, n_classes)

    def forward(self, input_ids, attention_mask, ontology_features):
        # 1. PhoBERT Encoding
        _, text_emb = self.phobert(input_ids=input_ids, attention_mask=attention_mask, return_dict=False)
        text_emb = self.dropout(text_emb)
        
        # 2. Fusion Logic
        if self.fusion_type == 'none':
            return self.fc(text_emb)
        
        # Xử lý vector KG
        ont_emb = self.ont_adapter(ontology_features)
        
        if self.fusion_type == 'gate':
            # Tính Gate từ văn bản (sigmoid -> 0..1)
            gate_values = torch.sigmoid(self.gate(text_emb))
            # Lọc tín hiệu KG
            ont_emb = ont_emb * gate_values
            
        # Nối vector (Concat)
        combined = torch.cat([text_emb, ont_emb], dim=1)
        
        return self.fc(combined)

print("✅ Model Architecture defined successfully!")

📏 Detected Ontology Dimension: 24
✅ Model Architecture defined successfully!


In [28]:
!pip install protobuf==3.20.3


In [29]:
# ---------------------------------------------------------
# CELL 5: TRAINER (FIXED IMPORTS)
# ---------------------------------------------------------
import copy
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score

# Đảm bảo device đã được define
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_experiment(model, name, epochs=15, patience=4):
    print(f"\n🚀 START: {name}")
    
    # Bọc DataParallel nếu có nhiều GPU
    if torch.cuda.device_count() > 1: 
        model = nn.DataParallel(model)
    model.to(device)
    
    optim = torch.optim.AdamW(model.parameters(), lr=2e-5)
    
    # Đảm bảo weights_tensor đã có (từ Cell 3). Nếu chưa thì dùng None (không cân bằng)
    if 'weights_tensor' in globals():
        crit = nn.CrossEntropyLoss(weight=weights_tensor)
    else:
        print("⚠️ Warning: 'weights_tensor' not found. Training without class weighting.")
        crit = nn.CrossEntropyLoss()
    
    best_f1, pat = 0, 0
    
    # Deepcopy weights ban đầu
    if isinstance(model, nn.DataParallel):
        best_w = copy.deepcopy(model.module.state_dict())
    else:
        best_w = copy.deepcopy(model.state_dict())
    
    for ep in range(epochs):
        model.train()
        loss_acc = 0
        
        # Train Loop
        for b in train_loader:
            optim.zero_grad()
            # Move batch to device
            ids = b['input_ids'].to(device)
            m = b['attention_mask'].to(device)
            o = b['ontology_features'].to(device)
            l = b['labels'].to(device)
            
            # Forward & Backward
            out = model(ids, m, o)
            loss = crit(out, l)
            loss.backward()
            optim.step()
            loss_acc += loss.item()
            
        # Validation
        acc, f1 = evaluate(model, valid_loader)
        avg_loss = loss_acc / len(train_loader)
        print(f"Ep {ep+1} | Loss: {avg_loss:.4f} | Val F1: {f1:.4f}", end=" ")
        
        # Early Stopping Check
        if f1 > best_f1:
            best_f1 = f1
            pat = 0
            if isinstance(model, nn.DataParallel):
                best_w = copy.deepcopy(model.module.state_dict())
            else:
                best_w = copy.deepcopy(model.state_dict())
            print("✅")
        else:
            pat += 1
            print(f"⚠️ {pat}")
        
        if pat >= patience:
            print(f"🛑 Early Stopping triggered after {ep+1} epochs.")
            break
    
    # Load Best Weights
    if isinstance(model, nn.DataParallel):
        model.module.load_state_dict(best_w)
    else:
        model.load_state_dict(best_w)
        
    return model

def evaluate(model, loader):
    model.eval()
    p, t = [], []
    with torch.no_grad():
        for b in loader:
            ids = b['input_ids'].to(device)
            m = b['attention_mask'].to(device)
            o = b['ontology_features'].to(device)
            l = b['labels'].to(device)
            
            out = model(ids, m, o)
            _, pr = torch.max(out, 1)
            p.extend(pr.cpu().numpy())
            t.extend(l.cpu().numpy())
            
    return accuracy_score(t, p), f1_score(t, p, average='macro')

In [30]:
# ---------------------------------------------------------
# CELL 6: MAIN EXPERIMENTS (5 CASES - VSFC)
# ---------------------------------------------------------
NC = 3 # Negative, Neutral, Positive

# 1. Baseline
m1 = PhoBERT_Fusion_V2(NC, 'none')
m1 = train_experiment(m1, "Baseline (PhoBERT)", epochs=15)

Some weights of RobertaModel were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



🚀 START: Baseline (PhoBERT)


KeyboardInterrupt: 

In [ ]:

# 2. Raw + Concat (xAI)
m2 = PhoBERT_Fusion_V2(NC, 'concat', ontology_dim=None)
m2 = train_experiment(m2, "Raw + Concat", epochs=15)

# 3. Raw + Gate
m3 = PhoBERT_Fusion_V2(NC, 'gate', ontology_dim=None)
m3 = train_experiment(m3, "Raw + Gate", epochs=15)



In [ ]:
# 4. Dense (64d) + Concat
m4 = PhoBERT_Fusion_V2(NC, 'concat', ontology_dim=64)
m4 = train_experiment(m4, "Dense(64) + Concat", epochs=15)

# 5. Dense (64d) + Gate
m5 = PhoBERT_Fusion_V2(NC, 'gate', ontology_dim=64)
m5 = train_experiment(m5, "Dense(64) + Gate", epochs=15)

In [ ]:
# ---------------------------------------------------------
# CELL 7: FINAL REPORT (FIXED)
# ---------------------------------------------------------
import pandas as pd
from sklearn.metrics import classification_report, f1_score, accuracy_score # <--- THÊM DÒNG NÀY

def gp(model, loader):
    model.eval()
    p, t = [], []
    with torch.no_grad():
        for b in loader:
            ids, m, o, l = [b[k].to(device) for k in ['input_ids','attention_mask','ontology_features','labels']]
            _, pr = torch.max(model(ids, m, o), 1)
            p.extend(pr.cpu().numpy()); t.extend(l.cpu().numpy())
    return t, p

# Dự đoán cho tất cả các model đã train
y_true, p1 = gp(m1, test_loader)
_, p2 = gp(m2, test_loader)
_, p3 = gp(m3, test_loader)
_, p4 = gp(m4, test_loader)
_, p5 = gp(m5, test_loader)

res = {
    "Model": ["Baseline", "Raw+Concat", "Raw+Gate", "Dense+Concat", "Dense+Gate"],
    "F1-Macro": [f1_score(y_true, p, average='macro') for p in [p1, p2, p3, p4, p5]],
    "Accuracy": [accuracy_score(y_true, p) for p in [p1, p2, p3, p4, p5]]
}

print("\n" + "="*50)
print("🏆 UIT-VSFC FINAL RESULTS")
print("="*50)
print(pd.DataFrame(res).round(4).to_markdown(index=False))

target_names = ["Negative", "Neutral", "Positive"] # 0, 1, 2
print("\n" + "="*50)
print("📑 DETAILED REPORT (DENSE + GATE)")
print("="*50)
# Bây giờ hàm này đã được định nghĩa và sẽ chạy bình thường
print(classification_report(y_true, p5, target_names=target_names, digits=4))

In [ ]:
# =========================================================
# CELL: PHOBERT DEEP FUSION (24 -> 64 -> 768)
# =========================================================
import torch
import torch.nn as nn
from transformers import AutoModel

class PhoBERT_DeepFusion(nn.Module):
    def __init__(self, n_classes, fusion_type='concat', ont_input_dim=24):
        super(PhoBERT_DeepFusion, self).__init__()
        self.fusion_type = fusion_type
        
        # 1. PhoBERT Backbone (768d)
        self.phobert = AutoModel.from_pretrained("vinai/phobert-base-v2")
        self.dropout = nn.Dropout(p=0.3)
        
        # 2. Deep Ontology Adapter: 24 -> 64 -> 768
        self.ont_adapter = nn.Sequential(
            nn.Linear(ont_input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 768), # Nâng lên cùng chiều với PhoBERT
            nn.BatchNorm1d(768),
            nn.ReLU()
        )
            
        # 3. Gating Mechanism (Nếu dùng Gate)
        if self.fusion_type == 'gate':
            # Gate học từ text (768) để điều tiết KG (768)
            self.gate = nn.Linear(768, 768)
            
        # 4. Classifier Head
        # Concat: 768 (Text) + 768 (KG) = 1536
        inp_dim = 768 + 768 
        self.fc = nn.Linear(inp_dim, n_classes)

    def forward(self, input_ids, attention_mask, ontology_features):
        # 1. PhoBERT Encoding
        _, text_emb = self.phobert(input_ids=input_ids, attention_mask=attention_mask, return_dict=False)
        text_emb = self.dropout(text_emb)
        
        # 2. Xử lý Deep KG Vector
        ont_emb = self.ont_adapter(ontology_features)
        
        # 3. Fusion Logic
        if self.fusion_type == 'gate':
            # Gating: Text quyết định lấy bao nhiêu phần KG
            gate_values = torch.sigmoid(self.gate(text_emb))
            ont_emb = ont_emb * gate_values
            
        # 4. Concat (Cả 'concat' và 'gate' sau khi lọc đều nối lại để giữ trọn thông tin)
        combined = torch.cat([text_emb, ont_emb], dim=1)
        
        return self.fc(combined)

# --- THỰC NGHIỆM VỚI 2 BIẾN THỂ MỚI ---

# 1. Deep Concat (24->64->768)
m_deep_concat = PhoBERT_DeepFusion(NC, 'concat')
m_deep_concat = train_experiment(m_deep_concat, "Deep(24-64-768) + Concat", epochs=15)

# 2. Deep Gate (24->64->768)
m_deep_gate = PhoBERT_DeepFusion(NC, 'gate')
m_deep_gate = train_experiment(m_deep_gate, "Deep(24-64-768) + Gate", epochs=15)

In [ ]:
# =========================================================
# CELL: THỰC NGHIỆM CHIỀU SÂU - DEEP BOTTLENECK (24 -> 64 -> 768)
# Kiến trúc này sử dụng một lớp ẩn 64 làm "nút thắt" để lọc nhiễu 
# trước khi nâng lên 768 chiều để Fusion với PhoBERT.
# =========================================================
import pandas as pd
import torch
from sklearn.metrics import f1_score, accuracy_score, classification_report

def run_deep_bottleneck_test():
    print(f"\n{'='*60}\n🔍 ĐANG KIỂM TRA HIỆU NĂNG KIẾN TRÚC DEEP (64)\n{'='*60}")
    
    # 1. Dự đoán trên tập Test cho Deep Concat (đã train ở Cell 88)
    # y_true được lấy từ labels của test_loader
    y_true, p_deep_concat = gp(m_deep_concat, test_loader)
    
    # 2. Dự đoán trên tập Test cho Deep Gate (đã train ở Cell 88)
    _, p_deep_gate = gp(m_deep_gate, test_loader)
    
    # 3. Tổng hợp kết quả
    results = {
        "Model Variant": ["Deep Concat (64d)", "Deep Gate (64d)"],
        "Ontology Path": ["24 -> 64 -> 768", "24 -> 64 -> 768"],
        "F1-Macro": [
            f1_score(y_true, p_deep_concat, average='macro'),
            f1_score(y_true, p_deep_gate, average='macro')
        ],
        "Accuracy": [
            accuracy_score(y_true, p_deep_concat),
            accuracy_score(y_true, p_deep_gate)
        ]
    }
    
    # Hiển thị bảng so sánh
    df_res = pd.DataFrame(results).round(4)
    print(df_res.to_markdown(index=False))
    
    # 4. In báo cáo chi tiết cho biến thể tốt nhất (thường là Deep Gate)
    target_names = ["Negative", "Neutral", "Positive"]
    print(f"\n📑 BÁO CÁO CHI TIẾT: DEEP GATE (64d)")
    print("-" * 30)
    print(classification_report(y_true, p_deep_gate, target_names=target_names, digits=4))

# Chạy hàm hiển thị kết quả
run_deep_bottleneck_test()

In [ ]:
class PhoBERT_WideFusion(nn.Module):
    def __init__(self, n_classes, fusion_type='concat', ont_input_dim=24):
        super(PhoBERT_WideFusion, self).__init__()
        self.fusion_type = fusion_type
        
        # 1. PhoBERT Backbone (768d)
        self.phobert = AutoModel.from_pretrained("vinai/phobert-base-v2")
        self.dropout = nn.Dropout(p=0.3)
        
        # 2. Wide Ontology Adapter: 24 -> 768 -> 768
        self.ont_pipeline = nn.Sequential(
            # Bước 1: Phóng chiếu rộng lên 768
            nn.Linear(ont_input_dim, 768),
            nn.BatchNorm1d(768),
            nn.ReLU(),
            
            # Bước 2: Xử lý sâu trong không gian 768
            nn.Linear(768, 768),
            nn.LayerNorm(768), # Dùng LayerNorm để ổn định không gian rộng
            nn.ReLU(),
            nn.Dropout(0.2)
        )
            
        # 3. Gating Mechanism (Nếu dùng Gate)
        if self.fusion_type == 'gate':
            # Cổng Gate 768x2 -> 768
            self.gate_fc = nn.Linear(768 * 2, 768)
            
        # 4. Classifier Head
        # Concat: 768 (BERT) + 768 (Wide KG) = 1536
        # Gate: 768
        inp_dim = 768 * 2 if self.fusion_type == 'concat' else 768
        self.fc = nn.Linear(inp_dim, n_classes)

    def forward(self, input_ids, attention_mask, ontology_features):
        # 1. PhoBERT Encoding
        _, text_emb = self.phobert(input_ids=input_ids, attention_mask=attention_mask, return_dict=False)
        text_emb = self.dropout(text_emb)
        
        # 2. Wide KG Vector Processing
        ont_emb = self.ont_pipeline(ontology_features)
        
        # 3. Fusion Logic
        if self.fusion_type == 'concat':
            combined = torch.cat([text_emb, ont_emb], dim=1)
            return self.fc(combined)
            
        elif self.fusion_type == 'gate':
            # Dynamic gating
            gate_input = torch.cat([text_emb, ont_emb], dim=1)
            z = torch.sigmoid(self.gate_fc(gate_input))
            fused_emb = (z * text_emb) + ((1 - z) * ont_emb)
            return self.fc(fused_emb)
        
        return self.fc(text_emb)

In [ ]:
# Cấu hình nhãn VSFC
NC = 3 

# 1. Wide Concat (24->768->768)
m_wide_concat = PhoBERT_WideFusion(NC, 'concat')
m_wide_concat = train_experiment(m_wide_concat, "Wide(24-768-768) + Concat", epochs=15)

# 2. Wide Gate (24->768->768)
m_wide_gate = PhoBERT_WideFusion(NC, 'gate')
m_wide_gate = train_experiment(m_wide_gate, "Wide(24-768-768) + Gate", epochs=15)

In [ ]:
# Lấy kết quả từ các model Deep đã có ở Cell 88
y_true, p_deep_concat = gp(m_deep_concat, test_loader)
_, p_deep_gate = gp(m_deep_gate, test_loader)

# L lấy kết quả từ các model Wide mới
_, p_wide_concat = gp(m_wide_concat, test_loader)
_, p_wide_gate = gp(m_wide_gate, test_loader)

final_res = {
    "Architecture": [
        "Baseline (No KG)", 
        "Deep Bottleneck (64d)", "Deep Bottleneck (64d)", 
        "Wide Projection (768d)", "Wide Projection (768d)"
    ],
    "Fusion": ["-", "Concat", "Gate", "Concat", "Gate"],
    "F1-Macro": [
        f1_score(y_true, p1, average='macro'), # p1 là baseline từ cell 86
        f1_score(y_true, p_deep_concat, average='macro'),
        f1_score(y_true, p_deep_gate, average='macro'),
        f1_score(y_true, p_wide_concat, average='macro'),
        f1_score(y_true, p_wide_gate, average='macro')
    ],
    "Accuracy": [
        accuracy_score(y_true, p1),
        accuracy_score(y_true, p_deep_concat),
        accuracy_score(y_true, p_deep_gate),
        accuracy_score(y_true, p_wide_concat),
        accuracy_score(y_true, p_wide_gate)
    ]
}

print("\n" + "="*60)
print("🏆 BẢNG SO SÁNH CÁC KIẾN TRÚC CHIỀU SÂU (VSFC)")
print("="*60)
df_final = pd.DataFrame(final_res).round(4)
print(df_final.to_markdown(index=False))
